In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import oracledb
from sqlalchemy import create_engine

import sys
import time
from datetime import datetime, timedelta

from mod_functions import mod_load_config

mod_config = mod_load_config()

In [ ]:
def get_timestamp():
    formatted_time = datetime.fromtimestamp(time.time())
    yea = formatted_time.year
    mon = formatted_time.month
    day = formatted_time.day
    hour = formatted_time.hour
    minute = formatted_time.minute


    htmp_tmstmp = f"{yea}-{mon}-{day}-{hour}-{minute}"
    return htmp_tmstmp

In [ ]:
def get_raw_df():
    connection=oracledb.connect(
        config_dir= mod_config["wallet_dir"],
        user=mod_config["oracle_user"],
        password=mod_config["oracle_password"],
        dsn = mod_config["dsn"],
        wallet_location=mod_config["wallet_location"],
        wallet_password=mod_config["wallet_password"])

    tn = mod_config["table_name"]
    sql = f"SELECT TO_CHAR(ut, 'YYYY-MM-DD HH24:MI') AS FORMATTED_UT, SYMLIST FROM {tn}"
    try:
        df = pd.read_sql(sql,connection)
    finally:
        connection.close()
    return df

In [ ]:
raw_df = get_raw_df()

In [ ]:
def process_df(df):
    df["SYMLIST"] = df["SYMLIST"].transform(eval)
    df = df.sort_values(by='FORMATTED_UT')
    df.reset_index(drop=True, inplace=True)
    df['FORMATTED_UT'] = pd.to_datetime(df['FORMATTED_UT'])


    today = datetime.now()

    tmdlt = today - timedelta(days=10)

    filtered_df = df[(df['FORMATTED_UT'] >= tmdlt) & (df['FORMATTED_UT'] <= today)]
    return filtered_df

In [ ]:
_df = process_df(raw_df)

In [ ]:
def get_dfhtmp():
    symPool = [i[1] for sublist in _df['SYMLIST'].values for i in sublist]
    unique_values = np.unique(symPool).tolist()

    dfhtmp = pd.DataFrame(columns=['FORMATTED_UT'] + unique_values)

    cols = dfhtmp.columns
    dfhtmp["FORMATTED_UT"] = _df["FORMATTED_UT"]
    inum = 1
    while inum < len(cols):
        sym = cols[inum]
        for index, row in _df.iterrows():
            for pair in row[1]:
                if sym == pair[1]:
                    dfhtmp.loc[index,sym] = pair[0]
        inum += 1  
    dfhtmp.set_index("FORMATTED_UT", inplace=True)
    dfhtmp = dfhtmp.fillna(0.0)
    return dfhtmp

In [ ]:
dfhtmp = get_dfhtmp()


In [ ]:
ddf= dfhtmp.describe()
ddfstd = ddf.sort_values(by='mean',axis='columns')

In [ ]:
dfhtmp_reordered = dfhtmp.reindex(columns=ddfstd.columns)

In [ ]:
dfhtmp_ds = dfhtmp_reordered.describe()


In [ ]:
dfhtmp_reordered

In [ ]:
df_max = dfhtmp_ds.loc['max'].max()
df_min = dfhtmp_ds.loc['max'][dfhtmp_ds.loc['max'] != 0].min()

htmp_tmstmp = get_timestamp()
htmp_title = f"HOTMAP  {htmp_tmstmp}"

sns.color_palette("mako", as_cmap=True)
plt.figure(figsize=(100, 40))
plt.title(htmp_title)

sns.heatmap(dfhtmp_reordered.T, annot=False, fmt=".0f", vmin = df_min, vmax = df_max, linewidths=.05, cbar_kws={'label': 'Your Colorbar Label'})
plt.savefig(f"hotmap-{htmp_tmstmp}.png")  
print(f"HOTMAP {htmp_tmstmp} SAVED")
# plt.show()


In [ ]:
# script_name = sys.argv[0]
# arguments = sys.argv[1:]
# date_start = arguments[0]  
# date_end = arguments[0]  